In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO
import os

In [ ]:
MODEL_PATH = "yolov8n.pt"  # n = nano (fast), s/m/l = more accurate
CONF_THRESHOLD = 0.4

MINI_TRUCK_MAX_AREA = 0.15  # tweak based on camera angle

In [ ]:
model = YOLO(MODEL_PATH)

# COCO class IDs
CAR_ID = 2
MOTORCYCLE_ID = 3
TRUCK_ID = 7


def classify_vehicle(class_id, bbox_area, frame_area):
    if class_id == MOTORCYCLE_ID:
        return "two_wheeler"

    if class_id == CAR_ID:
        return "car"

    if class_id == TRUCK_ID:
        if (bbox_area / frame_area) < MINI_TRUCK_MAX_AREA:
            return "mini_truck"
        else:
            return "truck"

    return None


def process_frame(frame):
    h, w = frame.shape[:2]
    frame_area = h * w

    counts = {
        "car": 0,
        "truck": 0,
        "mini_truck": 0,
        "two_wheeler": 0
    }

    results = model(frame, conf=CONF_THRESHOLD, verbose=False)

    for r in results:
        if r.boxes is None:
            continue

        for box in r.boxes:
            class_id = int(box.cls[0])
            x1, y1, x2, y2 = box.xyxy[0]
            bbox_area = (x2 - x1) * (y2 - y1)

            vehicle_type = classify_vehicle(class_id, bbox_area, frame_area)
            if vehicle_type:
                counts[vehicle_type] += 1

    return counts


def process_media(path):
    total_counts = {
        "car": 0,
        "truck": 0,
        "mini_truck": 0,
        "two_wheeler": 0
    }

    if path.lower().endswith((".jpg", ".png", ".jpeg")):
        frame = cv2.imread(path)
        return process_frame(frame)

    cap = cv2.VideoCapture(path)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_counts = process_frame(frame)
        for k in total_counts:
            total_counts[k] += frame_counts[k]

    cap.release()
    return total_counts

In [ ]:
input_df = pd.read_csv("input.csv")
output_rows = []

for _, row in input_df.iterrows():
    media_path = row["media_path"]

    if not os.path.exists(media_path):
        print(f"Missing file: {media_path}")
        continue

    counts = process_media(media_path)
    counts["id"] = row["id"]
    output_rows.append(counts)

output_df = pd.DataFrame(output_rows)
output_df = output_df[["id", "car", "truck", "mini_truck", "two_wheeler"]]
output_df.to_csv("output.csv", index=False)

print(" YOLOv8 processing complete. Saved to output.csv")